In [1]:
import os
import math
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer 
import tqdm
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch
from scipy.special import softmax

2024-03-23 13:32:18.830444: I tensorflow/core/platform/cpu_feature_guard.cc:181] Beginning TensorFlow 2.15, this package will be updated to install stock TensorFlow 2.15 alongside Intel's TensorFlow CPU extension plugin, which provides all the optimizations available in the package and more. If a compatible version of stock TensorFlow is present, only the extension will get installed. No changes to code or installation setup is needed as a result of this change.
More information on Intel's optimizations for TensorFlow, delivered as TensorFlow extension plugin can be viewed at https://github.com/intel/intel-extension-for-tensorflow.
2024-03-23 13:32:18.830494: I tensorflow/core/platform/cpu_feature_guard.cc:192] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [3]:
class Estimator():
    def __init__(self, tokenizer, model, num_labels, cuda = False, batch_size=256):
        
        TOKENIZER = tokenizer
        MODEL = model
        BATCH_SIZE = batch_size
        self.chunk_size = 0.2
        
        self.num_labels = num_labels
        self.tokenizer = AutoTokenizer.from_pretrained(TOKENIZER, use_fast=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=num_labels)
        self.cuda = cuda
        if cuda:
            self.model.cuda()
        
        self.training_args = TrainingArguments(  
            output_dir='./results',                   # output directory
            num_train_epochs=1,                  # total number of training epochs
            per_device_eval_batch_size=BATCH_SIZE,    # batch size for evaluation
        )

    def data_iterator(self, train_x, chunk_size = 500000):
        if chunk_size < 500000:
            chunk_size = 500000
        n_batches = math.ceil(len(train_x) / chunk_size)
        for idx in range(n_batches):
            x = train_x[idx *chunk_size:(idx+1) * chunk_size]
            yield x

    #eval_data is a list of input or a pandas frame
    def prepare_dataset(self, eval_data, max_length = 100):
        if type(eval_data) == list:
            print('start tokenizing %d lines of text'%len(eval_data))
            eval_encodings = self.tokenizer(eval_data, truncation=True, max_length = max_length, padding=True)
            eval_dataset = MyDataset(eval_encodings, [0]*len(eval_data))
            return eval_dataset
        
        
    def predict(self, eval_data, max_length = 100):      
        
        eval_iterator = self.data_iterator(eval_data, chunk_size=int(len(eval_data)*self.chunk_size))
        eval_preds = []
        
        for x in tqdm(eval_iterator):   
            trainer = Trainer(
                model=self.model,                         # the instantiated 🤗 Transformers model to be trained
                args=self.training_args,                       # training arguments, defined above
            )
            eval_dataset = self.prepare_dataset(x, max_length)
            eval_preds_raw, eval_labels , _ = trainer.predict(eval_dataset)
            if self.num_labels == 1:
                eval_preds += [it[0] for it in eval_preds_raw]
            else:
                print(np.argmax(eval_preds_raw, axis=-1))
                eval_preds += list(np.argmax(eval_preds_raw, axis=-1))
        
        return eval_preds

In [4]:
def get_toxicity(texts):
    os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
    os.environ["CUDA_VISIBLE_DEVICES"]="2"
    model = Estimator(tokenizer="s-nlp/roberta_toxicity_classifier", model="s-nlp/roberta_toxicity_classifier", num_labels=2, cuda = False)
    return model.predict(texts, max_length=128)

In [ ]:
file_list = os.listdir("/shared/4/projects/research-jam-2024/working-dir/monthly-posts-cleaned/")
for i in range(len(file_list) - 1, -1, -1):
  if "2009" in file_list[i]:
      print(file_list[i])
      file = os.path.join("/shared/4/projects/research-jam-2024/working-dir/monthly-posts-cleaned/", file_list[i])
      df = pd.read_csv(file, usecols=["message_id", "message_body_clean"], sep='\t')
      df = df.dropna()
      
      ids = df.message_id.to_list()
      content = df['message_body_clean'].to_list()
      pred = get_toxicity(content)
      
      df_toxicity = pd.DataFrame({"id":ids, "content":content, "pred":pred})
      new_filename = "/shared/4/projects/research-jam-2024/working-dir/toxicity/" + file_list[i] 
      df_toxicity.to_csv(new_filename, sep='\t', index=False)
      del df_toxicity, df, content, ids, pred

en.2009-03.tsv


/opt/anaconda/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of the model checkpoint at s-nlp/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (i

start tokenizing 500000 lines of text
